In [ ]:
from scipy.stats import genpareto, kendalltau, rankdata
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score
from dataclasses import dataclass, field
from sklearn.linear_model import Ridge
from joblib import Parallel, delayed
from datetime import datetime
from scipy.stats import ecdf
from arch import arch_model
from scipy.stats import qmc
import pyvinecopulib as pv
from fredapi import Fred
import yfinance as yf
import pandas as pd
import cvxpy as cp
import numpy as np
import time
import os

In [ ]:
class DataStore:
  def __init__(self, debug: bool = False):
    self.debug = debug

  def get_data(self, tickers: list, start: str, end: str, fred_api_key: str,
               interval: str = "1d", benchmark: str = "^GSPC",
               cyclical: str = "XLY", defensive: str = "XLP"):
    tmp = []
    for t in tickers:
      if isinstance(t, list):
        tmp.extend(t)
      elif isinstance(t, str):
        tmp.append(t)
      else:
        print(f"Warning: Skipping {t} | type: {type(t)}")
    tickers_clean = sorted(set(tmp))

    df_path = f"prices_{start}_{end}.parquet"
    if os.path.exists(df_path):
      df = pd.read_parquet(df_path)
    else:
      df = yf.download(tickers_clean, start=start, end=end, interval=interval)["Close"]
      df.to_parquet(df_path)

    returns = df.ffill().pct_change(fill_method=None)

    factors = self._get_factors(start, end, fred_api_key, benchmark, cyclical, defensive)
    return returns, factors

  def _fetch_equity_factors(self, start, end, benchmark="^GSPC",
                            cyclical="XLY", defensive="XLP"):
    r = yf.download([benchmark, "^IRX", cyclical, defensive],
                    start=start, end=end)["Close"]
    rets = r[[benchmark, cyclical, defensive]].pct_change(fill_method=None)
    rf_daily = (r["^IRX"].ffill() / 100) / 252

    factors = pd.DataFrame({
      "Benchmark": rets[benchmark] - rf_daily,
      "Spread": rets[cyclical] - rets[defensive],
    })
    return factors

  def _get_breakeven_inf(self, fred) -> pd.Series:
    return fred.get_series("DGS5") - fred.get_series("DFII5")

  def _fetch_fred(self, fred_api_key) -> pd.DataFrame:
    fred = Fred(api_key=fred_api_key)
    cols = {
      "NFCI":         fred.get_series("NFCI"),
      "USD":          fred.get_series("DTWEXBGS"),
      "TIPS10Y":      fred.get_series("DFII10"),
      "HighYieldOAS": fred.get_series("BAMLH0A0HYM2"),
      "BreakevenInf": self._get_breakeven_inf(fred),
    }
    out = pd.concat(cols, axis=1).sort_index()
    out.index = pd.to_datetime(out.index)
    return out

  def _get_factors(self, start, end, fred_api_key, benchmark="^GSPC",
                   cyclical="XLY", defensive="XLP") -> pd.DataFrame:
    path = f"factors_{start}_{end}.parquet"
    if os.path.exists(path):
      return pd.read_parquet(path)

    fred_df = self._fetch_fred(fred_api_key)
    eq_df = self._fetch_equity_factors(start, end, benchmark, cyclical, defensive)

    combined = pd.concat([fred_df, eq_df], axis=1, join="outer").sort_index()
    combined = combined.loc[start:end]
    combined.to_parquet(path)

    if self.debug:
      print("First valid date per factor:")
      print(combined.apply(pd.Series.first_valid_index))
    return combined

In [ ]:
@dataclass
class BootstrapParams:
  n_bootstraps: int = 200
  block_length: int = 10
  horizon: int = None
  circular: bool = False
  seed: int = None

  def __post_init__(self):
    if self.n_bootstraps < 1:
      raise ValueError(f"Invalid n_bootstraps: {self.n_bootstraps}")

    if self.block_length < 1:
      raise ValueError(f"Invalid block_length: {self.block_length}")

    if self.horizon is not None and self.horizon < 1:
      raise ValueError(f"Invalid horizon: {self.horizon}")

In [ ]:
class BlockBootstrap:
  def __init__(self, bootstrap_params: BootstrapParams = None, **kwargs):
    print("bp")
    self.p = bootstrap_params or BootstrapParams()
    self.rng = np.random.default_rng(self.p.seed)
    super().__init__(**kwargs)

  def _validate(self, returns: pd.DataFrame) -> int:
    if returns.isna().any().any():
      raise ValueError("returns contains NaNs; clean/align them first (e.g. dropna()).")

    T = len(returns)
    if self.p.block_length > T:
      raise ValueError(f"block_length ({self.p.block_length}) exceeds sample length ({T})")

    return self.p.horizon if self.p.horizon is not None else T

  def _sample_indices(self, T: int, horizon: int) -> np.ndarray:
    L = self.p.block_length
    n_blocks = int(np.ceil(horizon / L))

    n_starts = T if self.p.circular else T - L + 1
    starts = self.rng.integers(0, n_starts, size=(self.p.n_bootstraps, n_blocks))

    idx = (starts[:, :, None] + np.arange(L)).reshape(self.p.n_bootstraps, -1)
    idx = idx[:, :horizon]

    return idx % T if self.p.circular else idx

  def resample(self, returns: pd.DataFrame, as_dataframe: bool = False):
    horizon = self._validate(returns)
    values = returns.to_numpy()

    idx = self._sample_indices(len(values), horizon)
    samples = values[idx]

    if not as_dataframe:
      return samples

    index = pd.MultiIndex.from_product(
      [range(self.p.n_bootstraps), range(horizon)], names=["bootstrap", "step"]
    )
    return pd.DataFrame(
      samples.reshape(-1, samples.shape[2]), index=index, columns=returns.columns
    )

In [2]:
class TailWeightedKendall:
  _BLOCK_ELEMS = 2e7

  def __init__(self, params, **kwargs):
    self.p = params

  def empirical_uniforms(self, losses, T):
    losses = np.asarray(losses, dtype=float)
    if losses.ndim != 2:
      raise ValueError("losses must be two-dimensional.")

    k_ranks = rankdata(
        losses, method="average", axis=0
    )
    return k_ranks / (T + 1.0)

  def tail_weight(self, u):
    u = np.asarray(u, dtype=float)
    tail_depth = np.maximum(
        (u - self.p.tail_threshold) / (1 - self.p.tail_threshold),
        0.0
    ) ** self.p.tail_power

    return tail_depth


  def _get_tau(self, U, w):
    U = np.asarray(U, dtype=float)
    T_dim, N = U.shape
    z = (w <= 0).astype(float)

    G = np.zeros((N, N))
    D_and = np.zeros((N, N))
    D_or = np.zeros((N, N))
    Gw_and = np.zeros((N, N))
    Gw_or = np.zeros((N, N))
    zeros = np.zeros(N)

    block_size = max(1, int(self._BLOCK_ELEMS // (T_dim * N)))
    for a in range(0, T_dim, block_size):
      b = min(a + block_size, T_dim)
      S = np.sign(U[a:b, None, :] - U[None, :, :])
      S2 = S ** 2
      S_flat = S.reshape(-1, N)

      G += S_flat.T @ S_flat
      zeros += (S_flat == 0).sum(axis=0)

      W_pairwise = w[a:b, None, :] * w[None, :, :]
      A = S * W_pairwise

      A_flat = A.reshape(-1, N)
      Gw_and += A_flat.T @ A_flat
      S2W_flat = (S2 * W_pairwise).reshape(-1, N)
      D_and += 0.5 * (S2W_flat.T @ W_pairwise.reshape(-1, N))

      h_t = w[a:b, None, :]
      h_s = w[None, :, :]

      def wO(h):
        return h[..., :, None] + h[..., None, :] - h[..., :, None] * h[..., None, :]

      W = wO(h_s) * wO(h_t)
      mask = np.arange(w.shape[0])[None, :] < np.arange(a, b)[:, None]

      Gw_or += np.einsum('ts,tsij,tsi,tsj->ij', mask, W, S, S)
      D_or += np.einsum(
          'ts,tsij,tsi->ij',
          mask, W, S2
      )

    return G, Gw_and, Gw_or, zeros, D_and, D_or

  def _adjust_tau_or(self, tau_or_raw):
    m = (1 - self.p.tail_threshold) / (1 + self.p.tail_power)
    r = (1 - self.p.tail_threshold) / (2 + self.p.tail_power)

    tau_or_0 = -2 * ((1-2*r)/(2-m))**2

    return np.where(
        tau_or_raw >= tau_or_0,
        (tau_or_raw - tau_or_0) / (1.0 - tau_or_0),
        (tau_or_raw - tau_or_0) / (1.0 + tau_or_0)
    )


  def _tau_matrix(self, losses, U):
    T_dim, N = losses.shape
    h = self.tail_weight(U)
    n0 = T_dim * (T_dim - 1) / 2.0

    G, Gw_and, Gw_or, zeros, D_and, D_or_u = self._get_tau(losses, h)
    ties = (zeros - T_dim) / 2.0

    den_g = np.sqrt(np.outer(n0 - ties, n0 - ties))
    tau_glob= np.divide(G / 2.0, den_g, out=np.zeros((N, N)), where=den_g > 0)

    w2 = h ** 2
    W_prod = h.T @ h
    W2_prod = w2.T @ w2
    T_and = 0.5 * (W_prod ** 2 - W2_prod)
    D_v = D_and.T
    den_and = np.sqrt(D_and * D_v)

    tau_and = np.divide(
        0.5*Gw_and, den_and, out=np.full((N, N), np.nan), where=den_and > 0
    )

    ess_and = np.divide(
        W_prod ** 2,
        W2_prod,
        out=np.zeros((N, N)),
        where=W2_prod > 0
    )

    D_or_v = D_or_u.T

    den_or = np.sqrt(
        np.maximum(D_or_u, 0)
        * np.maximum(D_or_v, 0)
    )

    tau_or_raw = np.divide(
        Gw_or, den_or, out=np.full_like(Gw_or, np.nan), where=den_or > 0
    )

    s1 = h.sum(axis=0)
    h2 = h**2
    s2 = (h2).sum(axis=0)

    sum_w = s1[:, None] + s1[None, :] - h.T @ h
    sum_w2 = (
        s2[:, None] + s2[None, :]
        + h2.T @ h2 + 2*h.T @ h
        - 2*h2.T @ h - 2*(h2.T @ h).T
    )

    ess_or = np.divide(
        sum_w**2,
        sum_w2,
        out=np.zeros_like(sum_w),
        where=sum_w2 > 0
    )
    tau_or = self._adjust_tau_or(tau_or_raw)

    return tau_glob, tau_and, tau_or, ess_and, ess_or

  def _get_rel_score(self, ess):
    score = (
      (ess - self.p.min_effective_sample)
      / (ess + self.p.shrink_target_ess - self.p.min_effective_sample), 0.0, 1
    )

    score = np.clip(score, 0.0, 1.0)
    return score

  def fit_kendall(self, returns, assets=None, tau_prev=None, return_tau=False):
    if assets is None:
      assets = list(returns.columns)

    losses = -returns
    n_steps, n_assets = losses.shape

    U = self.empirical_uniforms(losses, T=n_steps)
    tau_glob, tau_and, tau_or, ess_and, ess_or = self._tau_matrix(losses=losses, U=U)


    reliability_score_and = self._get_rel_score(ess_and)
    reliability_score_or = self._get_rel_score(ess_or)

    eff_comp_and = self.p.max_tail_inf * self.p.and_pref * reliability_score_and
    eff_comp_or = self.p.max_tail_inf * (1 - self.p.and_pref) * reliability_score_or
    eff_comp_glob = 1 - eff_comp_and - eff_comp_or

    tau_final = (
        eff_comp_and * tau_and
        + eff_comp_or * tau_or
        + eff_comp_glob * tau_glob
    )

    return tau_final

  def get_tau_dist(self, returns, assets=None, tau_prev=None, return_tau=False):
      return self.fit_kendall(returns=returns, assets=assets,
                              tau_prev=tau_prev, return_tau=return_tau)


In [ ]:
@dataclass
class TailFit:
  c_L: float
  scale_L: float
  c_U: float
  scale_U: float
  p_l: float
  p_u: float
  u_lower: float
  u_upper: float
  ecdf_data: np.ndarray

In [ ]:
@dataclass
class CVaRHERCParams:
  vine_structure:np.ndarray=None
  tau_prev:np.ndarray=None
  vol_init:dict = field(default_factory=dict)
  r_init:dict = field(default_factory=dict)
  best_models:dict = field(default_factory=dict)
  best_fits:dict = field(default_factory=dict)
  best_lags:dict = field(default_factory=dict)
  marginal_distributions:dict = field(default_factory=dict)
  fit:bool=False

In [ ]:
class MarginalTailModel:
  def __init__(self, params, ticker):
    self.dist_p = params
    self.ticker = ticker
    self.tail_fit = None

  def _fit_gpd(self, exceed, side):
    c, _, scale = genpareto.fit(exceed, floc=0)

    xi_max = getattr(self.dist_p, "xi_max", None)
    if xi_max is not None and c > xi_max:
      c, _, scale = genpareto.fit(exceed, f0=xi_max, floc=0)

    if c >= 1.0:
      print(
        f"{self.ticker}: {side}-tail GPD xi={c:.4f} >= 1. \n"
        + "Infinite mean, so theoretical CVaR/ES is not finite.",
      )
    elif c >= 0.5:
      print(
        f"{self.ticker}: {side}-tail GPD xi={c:.4f} is very heavy. \n"
        + "CVaR can converge slowly even with QMC."
      )
    return c, scale

  def fit_dist(self, r):
    full_idx = r.index
    r = r.dropna()
    valid_idx = r.index
    x = r.to_numpy(dtype=float)
    n = x.size

    q = self.dist_p.cvar_q
    u_lower, u_upper = np.quantile(x, [1.0 - q, q])

    lower_mask = x < u_lower
    upper_mask = x > u_upper
    body_mask = ~(lower_mask | upper_mask)

    n_lo = int(lower_mask.sum())
    n_up = int(upper_mask.sum())
    if n_lo == 0 or n_up == 0:
      raise ValueError(f"{self.ticker}: insufficient observations in EVT tail.")

    exc_lo = u_lower - x[lower_mask]
    exc_up = x[upper_mask] - u_upper

    c_L, scale_L = self._fit_gpd(exc_lo, "lower")
    c_U, scale_U = self._fit_gpd(exc_up, "upper")

    p_l = n_lo / n
    p_u = n_up / n

    u = np.empty(n, dtype=float)

    u[lower_mask] = p_l * genpareto.sf(exc_lo, c_L, loc=0, scale=scale_L)

    body = x[body_mask]
    u[body_mask] = p_l + (1.0 - p_l - p_u) * (
      rankdata(body, method="average") / (body.size + 1)
    )

    u[upper_mask] = 1.0 - p_u * genpareto.sf(exc_up, c_U, loc=0, scale=scale_U)

    self.tail_fit = TailFit(
      c_L=c_L,
      scale_L=scale_L,
      c_U=c_U,
      scale_U=scale_U,
      p_l=p_l,
      p_u=p_u,
      u_lower=u_lower,
      u_upper=u_upper,
      ecdf_data=np.sort(body),
    )

    return pd.Series(u, index=valid_idx).reindex(full_idx)

  def inverse_cdf(self, u_resid):
    t = self.tail_fit
    u = np.asarray(u_resid, dtype=float)
    out = np.empty_like(u)

    lo = u < t.p_l
    hi = u > 1.0 - t.p_u
    body = ~(lo | hi)

    x = t.ecdf_data
    nb = x.size
    p_grid = t.p_l + (1.0 - t.p_l - t.p_u) * np.arange(1, nb + 1) / (nb + 1)
    out[body] = np.interp(u[body], p_grid, x)

    out[lo] = t.u_lower - genpareto.isf(
      u[lo] / t.p_l, t.c_L, loc=0, scale=t.scale_L
    )
    out[hi] = t.u_upper + genpareto.isf(
      (1.0 - u[hi]) / t.p_u, t.c_U, loc=0, scale=t.scale_U
    )
    return out

In [ ]:
class GarchFitter:
  @staticmethod
  def fit_arch(model, warm_start_params):
    fit_kwargs = dict(disp="off", show_warning=False)
    if warm_start_params is not None:
      try:
        return model.fit(
          starting_values=np.asarray(warm_start_params, dtype=float),
          **fit_kwargs,
        )
      except (ValueError, TypeError, np.linalg.LinAlgError):
        pass
    return model.fit(**fit_kwargs)

  @staticmethod
  def select_model(r, lag_candidates, o, warm_params):
    best_bic, best = np.inf, (None, None, None)
    for lags in lag_candidates:
      model = arch_model(
        r, mean="AR", lags=lags, vol="GARCH", p=1, o=o, q=1, dist="studentst"
      )
      try:
        fit = GarchFitter.fit_arch(model, warm_params)
      except (ValueError, np.linalg.LinAlgError):
        continue
      if fit.convergence_flag:
        continue

      pr = fit.params
      persistence = (
        pr["alpha[1]"] + pr["beta[1]"] + 0.5 * pr.get("gamma[1]", 0.0)
      )
      if persistence < 1 and fit.bic < best_bic:
        best_bic, best = fit.bic, (model, fit, lags)
    return best

  @staticmethod
  def fit_asset(r, max_lags, warm_params, warm_lags):
    if warm_lags is not None:
      res = GarchFitter.select_model(r, [warm_lags], 1, warm_params)
      if res[0] is not None:
        return res

    lags = range(1, max_lags + 1)
    res = GarchFitter.select_model(r, lags, 1, None)

    if res[0] is None:
      res = GarchFitter.select_model(r, lags, 0, None)

    if res[0] is None:
      raise ValueError("No stationary GARCH fit found")
    return res

In [1]:
class GarchEVTCopula(GarchFitter):
  def __init__(self, params, debug=False, **kwargs):
    self.debug = debug
    self.model_p = params
    self.sobol_seed = params.seed
    self._sobol_cache = None

  def _fit_ar_garch(self, returns):
    if self.debug:
      print("------------- Fitting AR-GARCH -------------")

    mf = self.model_fit
    cols = list(returns.columns)
    log_returns = np.log1p(returns)
    warm = bool(mf.fit)

    if not hasattr(mf, "ar_history"):
      mf.ar_history = {}

    series = {
      c: (log_returns[c] * self.model_p.scale_factor).dropna() for c in cols
    }

    jobs = []
    for c in cols:
      wp = wl = None
      if warm and c in mf.best_fits:
        wp, wl = mf.best_fits[c].params, mf.best_lags[c]
      jobs.append(delayed(GarchFitter._fit_asset)(series[c], self.model_p.lags, wp, wl))

    results = Parallel(n_jobs=getattr(self.model_p, "n_jobs", 1))(jobs)

    resid = {}
    for c, (model, fit, lags) in zip(cols, results):
      hist = series[c].to_numpy(dtype=float)
      if hist.size < lags:
        raise ValueError(
          f"{c}: need at least {lags} observations to initialize AR recursion."
        )
      mf.best_models[c] = model
      mf.best_fits[c] = fit
      mf.best_lags[c] = lags
      mf.ar_history[c] = hist[-lags:].copy()
      mf.vol_init[c] = float(np.asarray(fit.conditional_volatility)[-1])
      mf.r_init[c] = hist[-1]
      resid[c] = pd.Series(np.asarray(fit.std_resid), index=series[c].index)

    return pd.DataFrame(resid, index=returns.index, columns=cols, dtype=float)

  def _get_uniform_residuals(self, residuals):
    if self.debug:
      print("------------ Uniform Residuals -------------")

    u = {}
    for ticker in residuals.columns
      m = MarginalTailModel(self.model_p, ticker)
      u[ticker] = m.fit_dist(residuals[ticker])
      self.model_fit.marginal_distributions[ticker] = m
    return pd.DataFrame(u, index=residuals.index, columns=residuals.columns)

  def _fit_vine_copula(self, u_resid, structure=None):
    if self.debug:
      print("----------- Fitting Vine Copula ------------")

    clean = u_resid.dropna(how="any")
    if clean.empty:
      raise ValueError("No complete residual rows available for vine fit.")

    eps = 1e-10
    u = np.asfortranarray(
      np.clip(clean.to_numpy(dtype=np.float64), eps, 1 - eps)
    )

    kw = dict(num_threads=getattr(self.model_p, "vine_threads", 1))
    fam = getattr(self.model_p, "vine_families", None)
    trunc = getattr(self.model_p, "vine_trunc", None)

    if fam is not None:
      kw["family_set"] = fam
    if trunc is not None:
      kw["trunc_lvl"] = trunc

    controls = pv.FitControlsVinecop(**kw)
    self.copula = pv.Vinecop.from_data(
      u=u, controls=controls, structure=structure
    )

  def fit(self, returns, fit_params):
    self.model_fit = fit_params

    z_resid = self._fit_ar_garch(returns)
    self.cols = z_resid.columns
    self.start = z_resid.index[0]
    self.end = z_resid.index[-1]

    u_resid = self._get_uniform_residuals(z_resid)

    structure = self.model_fit.vine_structure if self.model_fit.fit else None
    self._fit_vine_copula(u_resid, structure=structure)

    self.model_fit.vine_structure = self.copula.structure
    self.model_fit.fit = True
    return self.model_fit

  def _base_uniforms(self, m, n_steps, n_assets):
    key = (m, n_steps, n_assets, self.sobol_seed)
    if self._sobol_cache is not None and self._sobol_cache[0] == key:
      return self._sobol_cache[1]

    engine = qmc.Sobol(d=n_steps * n_assets, scramble=True, seed=self.sobol_seed)
    u = np.clip(engine.random_base2(m=m), 1e-12, 1 - 1e-12)
    self._sobol_cache = (key, u)
    return u

  def _simulate(self, n_assets):
    n_steps = self.model_p.n_steps
    m = self.model_p.sobol_pow
    n = 2 ** m

    base = self._base_uniforms(m, n_steps, n_assets).reshape(n * n_steps, n_assets)
    u = self.copula.inverse_rosenblatt(
      np.ascontiguousarray(base),
      num_threads=getattr(self.model_p, "vine_threads", 1),
    )
    return np.asarray(u).reshape(n, n_steps, n_assets)

  def _inverse_semi_parametric_cdf(self, uniform_samples):
    if self.debug:
      print("-------- Inverse Semi Parametric CDF -------")

    n_paths, n_steps, n_assets = uniform_samples.shape
    z = np.empty((n_assets, n_steps, n_paths), dtype=float)
    eps = 1e-6

    for i, ticker in enumerate(self.cols):
      model = self.model_fit.marginal_distributions[ticker]
      u_i = np.clip(uniform_samples[:, :, i].T, eps, 1 - eps)
      z[i] = model.inverse_cdf(u_i)
    return z

  def _ar_coefficients(self, params, ticker, p):
    coeffs = np.empty(p, dtype=float)
    for lag in range(1, p + 1):
      key = f"{ticker}[{lag}]"
      if key not in params.index:
        raise KeyError(
          f"Expected AR parameter '{key}' not found. "
          f"Available parameters: {list(params.index)}"
        )
      coeffs[lag - 1] = float(params[key])
    return coeffs

  def _convert_to_returns(self, z):
    if self.debug:
      print("------------ Converting to Returns ---------")

    mf = self.model_fit
    A, T, N = z.shape
    lags = [int(mf.best_lags[c]) for c in self.cols]
    P = max(lags)

    const = np.zeros((A, 1))
    omega = np.zeros((A, 1))
    alpha = np.zeros((A, 1))
    gamma = np.zeros((A, 1))
    beta = np.zeros((A, 1))
    phi = np.zeros((A, P))
    sigma0 = np.zeros(A)
    eps0 = np.zeros(A)

    buf = np.zeros((A, P + T, N))

    for i, (c, p) in enumerate(zip(self.cols, lags)):
      fit = mf.best_fits[c]
      prm = fit.params
      const[i] = float(prm.get("Const", 0.0))
      omega[i] = float(prm["omega"])
      alpha[i] = float(prm["alpha[1]"])
      gamma[i] = float(prm.get("gamma[1]", 0.0))
      beta[i] = float(prm["beta[1]"])

      phi[i, P - p:] = self._ar_coefficients(prm, c, p)[::-1]

      hist = np.asarray(mf.ar_history[c], dtype=float)
      if hist.size != p:
        raise ValueError(f"{c}: AR history size {hist.size} != selected p={p}")
      buf[i, P - p:P, :] = hist[:, None]

      sigma0[i] = mf.vol_init[c]
      eps0[i] = float(np.asarray(fit.resid)[-1])

    sigma2 = np.repeat((sigma0 ** 2)[:, None], N, axis=1)
    eps_prev = np.repeat(eps0[:, None], N, axis=1)

    for t in range(T):
      sigma2 = omega + (alpha + gamma * (eps_prev < 0.0)) * eps_prev ** 2 + beta * sigma2
      np.maximum(sigma2, 1e-14, out=sigma2)

      eps_t = np.sqrt(sigma2) * z[:, t, :]
      ar = np.einsum("ap,apn->an", phi, buf[:, t:t + P, :])
      buf[:, P + t, :] = const + ar + eps_t
      eps_prev = eps_t

    return buf[:, P:, :]

  def generate_sample(self):
    if self.debug:
      print("------------ Generating Samples ------------")

    n_assets = len(self.cols)
    u = self._simulate(n_assets)
    z = self._inverse_semi_parametric_cdf(u)
    log_r = self._convert_to_returns(z) / self.model_p.scale_factor

    sim_r = np.ascontiguousarray(np.expm1(log_r).transpose(2, 0, 1))

    if not np.all(np.isfinite(sim_r)):
      raise FloatingPointError(
        "Simulation produced NaN/inf returns. "
        "Inspect EVT xi and GARCH path diagnostics."
      )
    return sim_r

In [ ]:
class CVaR(GarchEVTCopula):
  def __init__(
      self,
      params: ModelParams,
      debug: bool = False,
      **kwargs,
  ):
    super().__init__(
      debug=debug,
      params=params,
      **kwargs,
    )

    self.debug = debug
    self.cvar_p = params

    self.sobol_seed = params.seed
    self.sim_path = None

  def _make_sim_path(self):
    seed_label = (
      "random"
      if self.sobol_seed is None
      else str(self.sobol_seed)
    )

    return (
      f"simulation_{self.start}_{self.end}"
      f"_steps{self.model_p.n_steps}"
      f"_sobol2p{self.model_p.sobol_pow}"
      f"_ql{self.model_p.cvar_q:g}"
      f"_qu{self.model_p.cvar_q:g}"
      f"_seed{seed_label}.npy"
    )

  def fit_garch_evt_copula(self, returns, fit_params):
    self.start = returns.index[0]
    self.end = returns.index[-1]
    self.universe = returns.columns

    if not bool(fit_params.fit):
      fit_params = self.fit(returns, fit_params)
      self.sim_path = self._make_sim_path()
    else:
      self.model_fit = fit_params
      self.cols = returns.columns

    return fit_params

  def generate_r_sample(self, force_resimulate=False):
      if self.sim_path is None:
        raise ValueError("Fit the model first!")

      may_use_cache = (
        self.sobol_seed is not None
        and not force_resimulate
        and os.path.exists(self.sim_path)
      )

      if may_use_cache:
        return np.load(self.sim_path)

      sim_returns = self.generate_sample()

      if self.sobol_seed is not None:
        np.save(self.sim_path, sim_returns)

      return sim_returns

  def to_horizon_returns(self, R):
    R = np.asarray(R, dtype=float)
    if R.ndim == 3:
      return np.prod(1.0 + R, axis=2) - 1.0
    if R.ndim == 2:
      return R
    raise ValueError(f"Expected 2D or 3D returns, got {R.ndim}D.")

  def _tail_split(self, S, q):
    tail_mass = round((1.0 - q) * S, 9)
    k = int(np.floor(tail_mass))
    if k == 0:
      raise ValueError("Not enough scenarios for CVaR level.")
    return tail_mass, k, tail_mass - k

  def _asset_names(self, R, n_assets):
    if isinstance(R, pd.DataFrame):
      return list(R.columns)
    cols = getattr(self, "cols", None)
    if cols is not None and len(cols) == n_assets:
      return list(cols)
    return list(range(n_assets))

  def cvar(self, R, w=None, q=None):
    q = self.model_p.cvar_q if q is None else q
    names = self._asset_names(R, np.shape(R)[1])

    H = self.to_horizon_returns(R)

    S = H.shape[0]
    tail_mass, k, frac = self._tail_split(S, q)

    def tail_avg(L):
      Ls = -np.sort(-L, axis=0)
      total = Ls[:k].sum(axis=0)
      if frac > 0:
        total = total + frac * Ls[k]
      return total / tail_mass

    out = pd.Series(tail_avg(-H), index=names, name="CVaR")
    if w is not None:
      w = np.asarray(w, dtype=float).ravel()
      out["portfolio"] = tail_avg((-H @ w)[:, None])[0]
    return out

  def mcvar(self, R, w, q=None):
    q = self.model_p.cvar_q if q is None else q
    names = self._asset_names(R, np.shape(R)[1])

    H = self.to_horizon_returns(R)
    w = np.asarray(w, dtype=float).ravel()

    S = H.shape[0]
    tail_mass, k, frac = self._tail_split(S, q)

    order = np.argsort(-(-H @ w))
    weights = np.zeros(S)
    weights[order[:k]] = 1.0
    if frac > 0:
      weights[order[k]] = frac
    return pd.Series(weights @ (-H) / tail_mass, index=names, name="MCVaR")

  def verify_cvar(self, horizon_returns, w, b, scenario_prob=None):
    w = np.asarray(w, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()
    b = b / b.sum()

    mcvar = self.mcvar(horizon_returns, w)
    cvar = self.cvar(horizon_returns, w)["portfolio"]

    RC = w * mcvar.to_numpy()
    realized_budget = RC / cvar

    print("w:               ", w)
    print("MCVaR:           ", mcvar.to_numpy())
    print("RC:              ", RC)
    print("CVaR:            ", cvar)
    print("Euler sum:       ", RC.sum())
    print("target budget:   ", b)
    print("realized budget: ", realized_budget)
    print("Max budget error:", np.max(np.abs(realized_budget - b)))

    return mcvar, cvar

In [6]:
class HierarchicalFilter(TailWeightedKendall, CVaR):
  def __init__(self, params, **kwargs):
    self.fp = params
    super().__init__(params=params, **kwargs)

  def _get_kendall_distance(self, returns, tau_prev=None):
    dist_mtx = self.get_tau_dist(
        returns=returns,
        tau_prev=tau_prev,
        return_tau=False
    )

    return dist_mtx

  def _get_cvar(self, returns):
    R = returns.copy()
    return self.cvar(R)


  def _get_co_exceedances(self, returns):
    cvar = self._get_cvar(use_horizon = False)

    I = (returns <= cvar).astype(float)

    n_ij = I.T @ I
    n_i = I.sum(axis=0)

    denominator = np.sqrt(np.outer(n_i, n_i))

    lambda_hat = np.divide(n_ij, denominator, out=np.zeros_like(n_ij), where=denominator != 0)

    return 1 - lambda_hat


  def _get_ari_test_params(self, bootstrapped_returns, tau_prev):
      boot = np.asarray(bootstrapped_returns, dtype=float)

      n_boot, horizon, n_assets = boot.shape
      dists = np.empty((n_boot, n_assets, n_assets), dtype=float)
      linkages = []

      for b in range(n_boot):
        dist_b = self.get_tau_dist(
          returns=boot[b][:, :, None],
          tau_prev=tau_prev,
          return_tau=False
        ).to_numpy(dtype=float)

        dists[b] = dist_b
        linkages.append(
          linkage(squareform(dist_b, checks=False), method="average")
        )

      return {"dists": dists, "linkages": linkages}

    def _test_ari(self, k, params, ref_clusters):
      aris = [
        adjusted_rand_score(
          ref_clusters,
          fcluster(Z_b, t=k, criterion="maxclust")
        )
        for Z_b in params["linkages"]
      ]

      median = np.median(aris)
      ari_10_pct = np.quantile(aris, 0.1)

      return median, ari_10_pct


    def _get_opt_k(self, Z, bootstrapped_returns, tau_prev):
      ari_test_params = self._get_ari_test_params(
        bootstrapped_returns=bootstrapped_returns,
        tau_prev=tau_prev
      )

      dists = ari_test_params["dists"]
      linkages = ari_test_params["linkages"]
      n_boot, n_assets, _ = dists.shape

      if Z.shape[0] + 1 != n_assets:
        raise ValueError(
          "bootstrapped_returns and the reference linkage must have the same assets"
        )

      k_range = list(range(1, self.fp.max_k + 1))
      silhouette_scores = []
      ari_medians = []
      ari_10_pct_q = []

      for k in k_range:
        ref_clusters = fcluster(Z, t=k, criterion="maxclust")

        boot_silhouettes = []
        for b in range(n_boot):
          labels_b = fcluster(linkages[b], t=k, criterion="maxclust") - 1
          n_labels = len(np.unique(labels_b))

          if 2 <= n_labels < n_assets:
            boot_silhouettes.append(
              silhouette_score(dists[b], labels_b, metric="precomputed")
            )

        silhouette_scores.append(
          np.mean(boot_silhouettes) if boot_silhouettes else np.nan
        )


        median, ari_10_pct = self._test_ari(
          k=k,
          params=ari_test_params,
          ref_clusters=ref_clusters
        )

        ari_medians.append(median)
        ari_10_pct_q.append(ari_10_pct)

      ari_10_pct_q = np.asarray(ari_10_pct_q, dtype=float)
      ari_medians = np.asarray(ari_medians, dtype=float)
      silhouette_scores = np.asarray(silhouette_scores, dtype=float)

      k_valid = silhouette_scores[
          (ari_10_pct_q > self.fp.min_ari)
          & (silhouette_scores > self.fp.min_silhouette)
      ]

      if k_valid.size == 0:
        raise ValueError("No valid k found")

      optimal_k = k_valid.argmax() + 1


      return optimal_k


  def get_hierarchical_structure(self, returns, cos_sim, bootstrapped_returns, tau_prev=None):
    kendall_distance = None
    composite_distance = None
    cluster_labels = None

    factor_distance = np.sqrt((1 - cos_sim)/2)

    kendall_distance = self._get_kendall_distance(returns=returns, tau_prev=tau_prev)

    tail_dist = self._get_co_exceedances(returns)

    composite_distance = (
        self.factor_weight * factor_distance
        + self.kendall_weight * kendall_distance
        + self.tail_weight * tail_dist
    )

    comp_dist = squareform(composite_distance, checks=False)
    Z = linkage(comp_dist, method="average")

    optimal_k = self._get_opt_k(
      Z=Z,
      bootstrapped_returns=bootstrapped_returns,
      tau_prev=tau_prev
    )

    if optimal_k is not None:
      labels = fcluster(Z, t=optimal_k, criterion="maxclust")
      return pd.DataFrame({'Asset': composite_distance.index, 'Cluster': labels}), Z


    return (composite_distance, cluster_labels)




NameError: name 'CVaR' is not defined

In [ ]:
class Optimizer:
  def __init__(self, params, **kwargs):
    self.fp = params

  def _hedge_coverage_expr(z, hedge_score, p):
    N, S = hedge_score.shape
    H_s = []

    for s in range(S):
      h = hedge_score[:, s]

      levels = np.unique(
          np.r_[0.0, h]
      )

      coverage = 0
      for low, high in zip(levels[:-1], levels[1:]):
        eligible = np.flatnonzero(
            h >= high - 1e-12
        )

        coverage += (
            (high - low)
            * cp.minimum(
                1,
                cp.sum(
                    z[eligible]
                )
            )
        )

        H_s.append(coverage)

    return cp.sum(p * H_s)


  def optim_select_assets(self, unit, quality, r_hat, p_S, n_select, h_target=None):
    N_assets, K_factors = unit.shape
    I = np.identity(K_factors)
    z = cp.Variable(N_assets)
    h_scale = np.ones(N_assets) if h_target is None else h_target

    A = I + self.fp.factod_dir_importance * quality * cp.quad_form(unit, z)
    C_obj = cp.log_det(A)

    hedge_score = np.clip(
        r_hat / h_scale[None, :],
        0.0,
        1.0,
    )

    H_obj = self._hedge_coverage_expr(
        z,
        hedge_score,
        p_S
    )

    J_obj = C_obj + self.fp.hedge_theta * H_obj

    constraints = [
      z >= 0,
      z <= 1,
      cp.sum(z) == n_select
    ]

    obj = cp.Maximize()
    problem = cp.Problem(obj, constraints)

    problem.solve(solver=cp.CLARABEL)

    if problem.status not in (
      cp.OPTIMAL,
      cp.OPTIMAL_INACCURATE,
    ):
        raise RuntimeError(problem.status)

    return z.value, problem.value, A.value, H_obj.value




In [ ]:
class FactorFilter:
  def __init__(self, params: FilterParams, **kwargs):
    self.fp = params
    super().__init__(params=params, **kwargs)

  @staticmethod
  def _naive(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.index = pd.to_datetime(df.index).tz_localize(None)
    if not df.index.is_unique:
        raise ValueError("Time index must contain unique timestamps")
    return df.sort_index()

  def _compound_weekly(self, rets: pd.DataFrame) -> pd.DataFrame:
    if (rets.le(-1) & rets.notna()).any().any():
        raise ValueError("Compounded returns must be strictly greater than -1")
    g = np.log1p(rets).resample(self.fp.week_anchor)
    weekly = np.expm1(g.sum(min_count=1))
    counts = rets.resample(self.fp.week_anchor).count()
    return weekly.where(counts >= self.fp.min_days_per_week)

  def _to_weekly_returns(self, returns: pd.DataFrame) -> pd.DataFrame:
    return self._compound_weekly(rets=self._naive(df=returns))

  def _to_weekly_factors(self, factors: pd.DataFrame) -> pd.DataFrame:
    factors = self._naive(df=factors)
    kinds = self.fp.factor_kinds
    unknown = set(factors.columns) - set(kinds)

    if unknown:
        raise ValueError(f"No aggregation rule for factors: {sorted(unknown)}")
    supported = {"compound", "diff", "pct", "pct_change", "level"}
    invalid = {kinds[c] for c in factors if kinds[c] not in supported}

    if invalid:
        raise ValueError(f"Unsupported factor aggregation kind(s): {sorted(invalid)}")

    comp = [c for c in factors if kinds[c] == "compound"]
    levels = [c for c in factors if kinds[c] != "compound"]
    parts = []

    if comp:
      parts.append(self._compound_weekly(rets=factors[comp]))

    if levels:
        wk = factors[levels].resample(self.fp.week_anchor).last().ffill(limit=1)
        for c in levels:
          if kinds[c] == "diff":
              wk[c] = wk[c].diff()

          elif kinds[c] in {"pct", "pct_change"}:
              wk[c] = wk[c].pct_change(fill_method=None)

        parts.append(wk)

    if not parts:
        raise ValueError("At least one factor is required")
    return pd.concat(parts, axis=1).loc[:, factors.columns]

  def _time_horizon_filter(self, returns: pd.DataFrame) -> pd.DataFrame:
    if returns.empty:
        raise ValueError("No return observations")

    end = pd.to_datetime(returns.index.max())
    min_l, tp = self.fp.min_length

    if tp not in {"y", "m", "d"}:
        raise ValueError("min_length time unit must be y, m, or d")

    min_start = end - pd.DateOffset(
        years=min_l if tp == "y" else 0,
        months=min_l if tp == "m" else 0,
        days=min_l if tp == "d" else 0,
    )

    if self.fp.debug:
        print(f"Min start: {min_start}")

    keep = [
        asset for asset in returns
        if not returns[asset].dropna().empty
        and pd.to_datetime(returns[asset].dropna().index.min()) <= min_start
    ]

    return returns.loc[:, keep].dropna()

  def _fit_ridge(self, X: np.ndarray, y: np.ndarray):
      return Ridge(alpha=self.fp.ridge_alpha, fit_intercept=True).fit(X, y)

  def _rolling_factor_params(self, returns: pd.DataFrame, factors: pd.DataFrame):
    idx = returns.index.intersection(factors.index).sort_values()
    returns, factors = returns.loc[idx], factors.loc[idx]

    if returns.isna().any().any() or factors.isna().any().any():
        raise ValueError("Missing data in regression inputs")

    X, Y = factors.to_numpy(dtype=float), returns.to_numpy(dtype=float)
    T, N = Y.shape
    K = X.shape[1]
    w = self.fp.window_size

    if T <= w:
        raise ValueError(f"Need more than {w} weekly observations; got {T}")

    betas_raw = np.full((T, N, K), np.nan)
    betas_std = np.full((T, N, K), np.nan)
    delta_r2 = np.full((T, N, K), np.nan)
    r2_in = np.full((T, N), np.nan)
    r2_oos = np.full((T, N), np.nan)
    rmse_oos = np.full((T, N), np.nan)
    resid_vol_in = np.full((T, N), np.nan)
    oos_resid = np.full((T, N), np.nan)

    cumulative_sse = np.zeros(N)
    cumulative_sst = np.zeros(N)
    cumulative_sse_wo = np.zeros((K, N))

    def safe_r2(sse, sst):
      ratio = np.full_like(sse, np.nan, dtype=float)
      np.divide(sse, sst, out=ratio, where=sst > 0)
      return 1.0 - ratio

    def fit_window(start, stop):
      x_train, y_train = X[start:stop], Y[start:stop]
      scaler = StandardScaler()
      x_scaled = scaler.fit_transform(x_train)
      model = self._fit_ridge(X=x_scaled, y=y_train)
      train_resid = y_train - model.predict(x_scaled)
      y_mean = y_train.mean(axis=0)
      train_sse = np.sum(train_resid**2, axis=0)
      train_sst = np.sum((y_train - y_mean)**2, axis=0)
      return (
          model, scaler, safe_r2(sse=train_sse, sst=train_sst),
          train_resid.std(axis=0, ddof=0), y_mean,
      )

    for t in range(w, T):
      model, scaler, train_r2, train_vol, y_mean = fit_window(start=t - w, stop=t)
      betas_std[t - 1] = model.coef_
      betas_raw[t - 1] = model.coef_ / scaler.scale_[None, :]
      r2_in[t - 1], resid_vol_in[t - 1] = train_r2, train_vol

      x_test = scaler.transform(X[t:t + 1])
      prediction = model.predict(x_test)[0]
      error = Y[t] - prediction
      oos_resid[t] = error
      cumulative_sse += error**2
      cumulative_sst += (Y[t] - y_mean)**2

      x_win, y_win = scaler.transform(X[t - w:t]), Y[t - w:t]

      for k in range(K):
        if K > 1:
          x_wo = np.delete(x_win, k, axis=1)
          model_wo = self._fit_ridge(X=x_wo, y=y_win)
          pred_wo = model_wo.predict(np.delete(x_test, k, axis=1))[0]

        else:
          pred_wo = y_mean
        cumulative_sse_wo[k] += (Y[t] - pred_wo)**2

      n_oos = t - w + 1
      r2_oos[t] = safe_r2(sse=cumulative_sse, sst=cumulative_sst)
      rmse_oos[t] = np.sqrt(cumulative_sse / n_oos)
      delta = np.full((K, N), np.nan)
      np.divide(
          cumulative_sse_wo - cumulative_sse,
          cumulative_sst,
          out=delta,
          where=cumulative_sst > 0,
      )
      delta_r2[t] = delta.T

    model, scaler, train_r2, train_vol, _ = fit_window(start=T - w, stop=T)
    betas_std[-1] = model.coef_
    betas_raw[-1] = model.coef_ / scaler.scale_[None, :]
    r2_in[-1], resid_vol_in[-1] = train_r2, train_vol

    residuals_df = pd.DataFrame(oos_resid, index=idx, columns=returns.columns)
    resid_vol_oos = (
        residuals_df.expanding(min_periods=2).std(ddof=0).to_numpy()
    )

    beta_columns = pd.MultiIndex.from_product(
        [returns.columns, factors.columns], names=["Asset", "Factor"]
    )

    def make_beta_df(values):
      return pd.DataFrame(
          values.reshape(T, N * K), index=idx, columns=beta_columns
      )

    metrics = ["R2_in", "R2_oos", "RMSE_oos", "ResidVol_in", "ResidVol_oos"]
    stats_values = np.stack(
        [r2_in, r2_oos, rmse_oos, resid_vol_in, resid_vol_oos], axis=2
    )

    stats_df = pd.DataFrame(
        stats_values.reshape(T, N * len(metrics)),
        index=idx,
        columns=pd.MultiIndex.from_product(
            [returns.columns, metrics], names=["Asset", "Metric"]
        ),
    )

    return (
        make_beta_df(values=betas_raw),
        make_beta_df(values=betas_std),
        make_beta_df(values=delta_r2),
        stats_df,
        residuals_df,
    )

  @staticmethod
  def _shrink_vasicek(
    betas_raw: pd.DataFrame,
    est_var: pd.DataFrame,
  ) -> tuple[pd.DataFrame, pd.DataFrame]:
    est_var = est_var.reindex(index=betas_raw.index, columns=betas_raw.columns)

    if est_var.isna().any().any() or (est_var < 0).any().any():
        raise ValueError("est_var must contain nonnegative (N,K) estimates")

    prior_mean = betas_raw.mean(axis=0)

    prior_var = (
        betas_raw.var(axis=0, ddof=1).fillna(0.0)
        - est_var.mean(axis=0)
    ).clip(lower=0.0)

    numerator = np.broadcast_to(prior_var.to_numpy()[None, :], betas_raw.shape)
    denominator = numerator + est_var.to_numpy()

    weights = np.divide(
        numerator, denominator,
        out=np.zeros_like(denominator), where=denominator > 0,
    )

    w_raw = pd.DataFrame(weights, index=betas_raw.index, columns=betas_raw.columns)
    adjusted = w_raw * betas_raw + (1.0 - w_raw) * prior_mean

    return adjusted, w_raw

  @staticmethod
  def _sqrt_psd(cov: np.ndarray) -> np.ndarray:
    cov = np.asarray(cov, dtype=float)

    if cov.ndim not in (2, 3) or cov.shape[-1] != cov.shape[-2]:
        raise ValueError("cov must be (K,K) or (B,K,K)")

    cov = (cov + np.swapaxes(cov, -1, -2)) / 2.0
    eigval, eigvec = np.linalg.eigh(cov)
    root = np.sqrt(np.clip(eigval, 0.0, None))

    return (eigvec * root[..., None, :]) @ np.swapaxes(eigvec, -1, -2)

  def _risk_exposure_similarity(
    self, betas: pd.DataFrame, factors: pd.DataFrame,
  ):
    if isinstance(betas, pd.Series):
        betas = betas.unstack("Factor")

    if not isinstance(betas, pd.DataFrame) or isinstance(betas.columns, pd.MultiIndex):
        raise ValueError("betas must be (N,K), indexed by Asset and columns by Factor")

    missing = [c for c in factors.columns if c not in betas.columns]

    if missing:
        raise ValueError(f"No betas for factors: {missing}")

    betas = betas.loc[:, factors.columns]
    sigma = factors.dropna().cov().loc[factors.columns, factors.columns].to_numpy()
    sqrt_sigma = self._sqrt_psd(cov=sigma)
    exposure = betas.to_numpy(dtype=float) @ sqrt_sigma
    norms = np.linalg.norm(exposure, axis=1, keepdims=True)

    if np.any(norms <= 1e-12):
        raise ValueError("Cannot normalize an asset with zero factor exposure")

    U = exposure / norms
    cos = np.clip(U @ U.T, -1.0, 1.0)
    exposures = pd.DataFrame(exposure, index=betas.index, columns=factors.columns)

    unit = pd.DataFrame(U, index=betas.index, columns=factors.columns)
    cos_sim = pd.DataFrame(cos, index=betas.index, columns=betas.index)
    cos_sim.index.name = cos_sim.columns.name = "Asset"

    return exposures, unit, cos_sim

  def _get_h_ref(
    self,
    betas: np.ndarray,
    factor_cov: np.ndarray,
    factor_scenarios: np.ndarray,
    a: np.ndarray,
    scenario_idx=None,
    i_idx=None,
    j_idx=None,
  ) -> np.ndarray:
    beta = np.asarray(betas, dtype=float)

    if beta.ndim != 2:
        raise ValueError("betas must have shape (N,K)")

    N, K = beta.shape
    delta = np.asarray(factor_scenarios, dtype=float)
    scale = np.asarray(a, dtype=float)
    sigma = np.asarray(factor_cov, dtype=float)

    if delta.ndim != 2 or delta.shape[1] != K or sigma.shape != (K, K):
        raise ValueError("factor_scenarios (S,K), betas (N,K), covariance (K,K) required")

    if scale.shape != (len(delta),):
        raise ValueError("a must have shape (S,)")

    if scenario_idx is not None:
        s = np.asarray(scenario_idx, dtype=int)
        delta, scale = delta[s], scale[s]

    if not np.isfinite(scale).all() or np.any(scale <= 0):
        raise ValueError("Scenario scales must be positive and finite")

    i_idx = np.arange(N) if i_idx is None else np.atleast_1d(i_idx).astype(int)
    j_idx = np.arange(N) if j_idx is None else np.atleast_1d(j_idx).astype(int)

    mag = np.linalg.norm(beta @ self._sqrt_psd(cov=sigma), axis=1)

    ratio = np.divide(
        mag[i_idx, None], mag[None, j_idx],
        out=np.zeros((len(i_idx), len(j_idx))),
        where=mag[None, j_idx] > 1e-12,
    )

    proj = beta[j_idx] @ delta.T
    h = ratio[:, :, None] * proj[None, :, :] / scale[None, None, :]

    return np.clip(h, 0.0, 1.0)

  def _get_ref_redundancy(
      self,
      unit: pd.DataFrame,
      A_prev: np.ndarray,
      H: np.ndarray,
      quality: np.ndarray,
      prev_mask: np.ndarray,
      new_idx: int,
      h_ref: np.ndarray,
      p_s: np.ndarray,
      cost: np.ndarray | None = None,
  ) -> float:
    old = np.flatnonzero(prev_mask)
    if old.size == 0:
        return 0.0

    U = unit.to_numpy(dtype=float)
    N, K = U.shape
    q = np.asarray(quality, dtype=float)

    if q.shape != (N,) or A_prev.shape != (K, K):
        raise ValueError("quality (N,), A_prev (K,K) required")

    H = np.asarray(H, dtype=float)
    p_s = np.asarray(p_s, dtype=float)
    h = np.asarray(h_ref, dtype=float)

    if h.shape != (len(old), len(H)) or p_s.shape != H.shape:
        raise ValueError("h_ref must be (n_previous,S), and H/p_s must be (S,)")

    kappa = self.fp.factor_dir_importance
    theta = self.fp.theta
    lam_c = self.fp.lambda_cost

    U_j = U[old]
    quad = np.einsum("jk,kj->j", U_j, np.linalg.solve(A_prev, U_j.T))

    info = np.log1p(kappa * q[new_idx] * quad)
    scen = theta * (np.maximum(0.0, h - H[None, :]) @ p_s)
    c_i = 0.0 if cost is None else float(np.asarray(cost)[new_idx])

    return float((info + scen - lam_c * c_i).max())

  def _get_excess_coverage(
      self,
      unit: pd.DataFrame,
      betas: np.ndarray,
      factor_cov: np.ndarray,
      quality: np.ndarray,
      factor_scenarios: np.ndarray,
      scenario_scale: np.ndarray,
      p_s: np.ndarray,
      prev_result,
      current_result,
      cost: np.ndarray | None = None,
  ) -> tuple[float, float, float]:
    _, J_prev, prev_mask, A_prev, H_prev = prev_result
    _, J_curr, selected_mask, _, _ = current_result
    new = np.flatnonzero(selected_mask & ~prev_mask)
    removed = np.flatnonzero(prev_mask & ~selected_mask)

    if len(new) != 1 or len(removed) != 0:
        raise ValueError(
            "_select_assets must return nested selected sets; "
            "one new asset with no removals is required"
        )

    new_idx = int(new[0])
    old = np.flatnonzero(prev_mask)

    if len(old):
        h_ref = self._get_h_ref(
            betas=betas,
            factor_cov=factor_cov,
            factor_scenarios=factor_scenarios,
            a=scenario_scale,
            i_idx=[new_idx],
            j_idx=old,
        )[0]

    else:
        h_ref = np.empty((0, len(p_s)), dtype=float)

    redundancy = self._get_ref_redundancy(
        unit=unit,
        A_prev=A_prev,
        H=H_prev,
        quality=quality,
        prev_mask=prev_mask,
        new_idx=new_idx,
        h_ref=h_ref,
        p_s=p_s,
        cost=cost,
    )

    marginal_gain = J_curr - J_prev

    return float(marginal_gain - redundancy), float(redundancy), float(marginal_gain)

In [ ]:
class RegimeFilter:
    def __init__(self, params: FilterParams, **kwargs):
      super().__init__(params=params, **kwargs)

In [ ]:
class Filter(HierarchicalFilter, RegimeFilter, FactorFilter, BlockBootstrap, Optimizer):
    def __init__(
      self,
      filter_params: FilterParams,
      bootstrap_params: BootstrapParams,
    ):
      super().__init__(params=filter_params, bootstrap_params=bootstrap_params)

    def _get_bootstraps(self, returns: pd.DataFrame, factors: pd.DataFrame):
      common = returns.dropna().index.intersection(factors.dropna().index)
      Y_df, X_df = returns.loc[common], factors.loc[common]
      if len(common) < 2:
          raise ValueError("Need at least two aligned weekly observations")

      assets, fnames = Y_df.columns, X_df.columns
      N, K = len(assets), len(fnames)

      draws = np.asarray(
          self.resample(returns=pd.concat([Y_df, X_df], axis=1)), dtype=float
      )

      if draws.ndim != 3 or draws.shape[2] != N + K:
          raise ValueError("resample must return (B, W, N+K) in input column order")
      B, W, _ = draws.shape

      if B < 2 or W < 2:
          raise ValueError("Need B>=2 bootstrap draws with W>=2 observations each")

      if not np.isfinite(draws).all():
          raise ValueError("Bootstrap draws contain non-finite values")

      Yb = draws[:, :, :N]
      Xb = draws[:, :, N:].copy()

      coefs = np.empty((B, N, K), dtype=float)

      for b in range(B):
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(Xb[b])
        coefs[b] = (
            self._fit_ridge(X=X_scaled, y=Yb[b]).coef_
            / scaler.scale_[None, :]
        )

      lower, upper = np.percentile(coefs, [2.5, 97.5], axis=0)
      p_pos = (coefs > 0).mean(axis=0)

      stability = np.maximum(p_pos, 1 - p_pos)
      factor_index = pd.Index(fnames, name="Factor")

      percentile_intervals = pd.DataFrame(
          np.stack([lower, upper], axis=2).transpose(1, 0, 2).reshape(K, N * 2),
          index=factor_index,
          columns=pd.MultiIndex.from_product(
              [assets, ["Lower CI (2.5%)", "Upper CI (97.5%)"]],
              names=["Asset", "Bound"],
          ),
      )

      sign_stability = pd.DataFrame(
          stability.T, index=factor_index,
          columns=pd.Index(assets, name="Asset"),
      )

      beta_estimation_var = coefs.var(axis=0, ddof=1)

      return (
        percentile_intervals,
        sign_stability,
        p_pos,
        beta_estimation_var,
        coefs,
        Xb,
      )

    def _get_factor_exposure_features(
      self,
      returns: pd.DataFrame,
      factors: pd.DataFrame,
    ):
      weekly_returns = self._to_weekly_returns(returns=returns).dropna()
      weekly_factors = self._to_weekly_factors(factors=factors).dropna()
      common = weekly_returns.index.intersection(weekly_factors.index).sort_values()
      X, Y = weekly_factors.loc[common], weekly_returns.loc[common]

      if len(common) <= self.fp.window_size:
          raise ValueError("Insufficient aligned weekly data for rolling estimation")

      w = self.fp.window_size
      (
          percentile_intervals, sign_stability, p_pos,
          beta_estimation_var, coefs, factor_bootstrap,
      ) = self._get_bootstraps(returns=Y.iloc[-w:], factors=X.iloc[-w:])

      betas_raw, betas_std, delta_r2, stats_df, residuals = (
          self._rolling_factor_params(returns=Y, factors=X)
      )
      assets, fnames = Y.columns, X.columns
      est_var = pd.DataFrame(beta_estimation_var, index=assets, columns=fnames)

      latest_raw = betas_raw.iloc[-1].unstack("Factor").reindex(
          index=assets, columns=fnames
      )

      beta_adjusted, _ = self._shrink_vasicek(betas_raw=latest_raw, est_var=est_var)

      lag = int(self.fp.beta_change_window)

      if lag < 1:
          raise ValueError("beta_change_window must be >= 1")

      if len(betas_raw) > lag and betas_raw.iloc[-1 - lag].notna().all():
          old_raw = betas_raw.iloc[-1 - lag].unstack("Factor").reindex(
              index=assets, columns=fnames
          )
          old_adjusted, _ = self._shrink_vasicek(
              betas_raw=old_raw, est_var=est_var
          )
          beta_change = beta_adjusted - old_adjusted

      else:
          beta_change = pd.DataFrame(np.nan, index=assets, columns=fnames)

      last_delta = delta_r2.iloc[-1].unstack("Factor").reindex(
          index=assets, columns=fnames
      )

      factor_part = pd.concat(
        {
          f: pd.DataFrame(
            {
              "Lower CI (2.5%)": percentile_intervals.loc[f].xs(
                "Lower CI (2.5%)", level="Bound"
              ).reindex(assets),
              "Upper CI (97.5%)": percentile_intervals.loc[f].xs(
                "Upper CI (97.5%)", level="Bound"
              ).reindex(assets),
              "SignStability": sign_stability.loc[f].reindex(assets),
              "DeltaR2_oos": last_delta[f],
            }, index=assets,
          )
          for f in fnames
        }, axis=1,
      )
      model_cols = ["R2_in", "R2_oos", "RMSE_oos", "ResidVol_in", "ResidVol_oos"]
      latest_stats = stats_df.iloc[-1].unstack("Metric").reindex(index=assets)
      model_part = pd.concat({"Model": latest_stats[model_cols]}, axis=1)
      summary = pd.concat([factor_part, model_part], axis=1)
      summary.columns.names = ["Factor", "Field"]
      summary.index.name = "Asset"

      return (
          beta_adjusted, beta_change, summary,
          coefs, factor_bootstrap, X.iloc[-w:], est_var,
      )

    @staticmethod
    def _validate_selection_result(result, N: int, K: int, S: int, m: int):
      if len(result) != 5:
          raise ValueError("_select_assets must return (weights, J, mask, A, H)")

      weights, J, mask, A, H = result
      weights = np.asarray(weights, dtype=float)
      mask = np.asarray(mask, dtype=bool)
      A = np.asarray(A, dtype=float)
      H = np.asarray(H, dtype=float)

      if weights.shape != (N,) or mask.shape != (N,):
          raise ValueError("Optimizer weights/mask must both be (N,)")

      if A.shape != (K, K) or H.shape != (S,):
          raise ValueError("Optimizer A must be (K,K) and H must be (S,)")

      if np.count_nonzero(mask) != m:
          raise ValueError(f"Optimizer returned {mask.sum()} assets; requested {m}")

      if not np.isfinite(J) or not (np.isfinite(A).all() and np.isfinite(H).all()):
          raise ValueError("Optimizer returned non-finite results")

      return weights, float(J), mask, A, H

    def _get_uncertainty(
      self,
      coefs: np.ndarray,
      factors_boot: np.ndarray,
      unit: pd.DataFrame,
      quality: np.ndarray,
      m: int,
      factor_scenarios: np.ndarray,
      scenario_scale: np.ndarray,
      p_s: np.ndarray,
      beta_estimation_var: pd.DataFrame | None = None,
    ) -> float:
      coefs = np.asarray(coefs, dtype=float)
      factors_boot = np.asarray(factors_boot, dtype=float)

      if coefs.ndim != 3 or factors_boot.ndim != 3:
          raise ValueError("coefs (B,N,K) and factors_boot (B,W,K) required")

      B, N, K = coefs.shape
      if B < 2 or factors_boot.shape[0] != B or factors_boot.shape[2] != K:
          raise ValueError("Incompatible bootstrap draws (B,N,K) and (B,W,K)")

      if factors_boot.shape[1] < 2 or not 2 <= m <= N:
          raise ValueError("Need W>=2 and m in [2,N]")

      if unit.shape != (N, K) or quality.shape != (N,):
          raise ValueError("unit (N,K) and quality (N,) must match coefs")

      Xc = factors_boot - factors_boot.mean(axis=1, keepdims=True)
      cov_b = np.einsum("bti,btj->bij", Xc, Xc) / (factors_boot.shape[1] - 1)
      sqrt_b = self._sqrt_psd(cov=cov_b)
      excess = np.empty(B, dtype=float)
      S = len(p_s)
      for b in range(B):
          beta_b = pd.DataFrame(coefs[b], index=unit.index, columns=unit.columns)

          if beta_estimation_var is not None:
              beta_b, _ = self._shrink_vasicek(
                  betas_raw=beta_b, est_var=beta_estimation_var
              )

          beta_array = beta_b.to_numpy(dtype=float)
          exposure_b = beta_array @ sqrt_b[b]
          norms = np.linalg.norm(exposure_b, axis=1, keepdims=True)

          if np.any(norms <= 1e-12):
              raise ValueError(f"Bootstrap {b} contains zero factor exposure")

          unit_b = pd.DataFrame(
              exposure_b / norms, index=unit.index, columns=unit.columns
          )

          r_hat_b = beta_array @ factor_scenarios.T
          prev = self._validate_selection_result(
              result=self.optim_select_assets(
                  unit=unit_b, quality=quality, r_hat=r_hat_b, n_select=m - 1
              ),
              N=N, K=K, S=S, m=m - 1,
          )

          current = self._validate_selection_result(
              result=self.optim_select_assets(
                  unit=unit_b, quality=quality, r_hat=r_hat_b, n_select=m
              ),
              N=N, K=K, S=S, m=m,
          )

          excess[b], _, _ = self._get_excess_coverage(
              unit=unit_b,
              betas=beta_array,
              factor_cov=cov_b[b],
              quality=quality,
              factor_scenarios=factor_scenarios,
              scenario_scale=scenario_scale,
              p_s=p_s,
              prev_result=prev,
              current_result=current,
          )

      return float(excess.std(ddof=1))

    def _select_opt_n_assets(
      self,
      beta_adjusted: pd.DataFrame,
      unit: pd.DataFrame,
      quality: np.ndarray,
      coefs: np.ndarray,
      factor_scenarios: np.ndarray,
      scenario_scale: np.ndarray,
      factor_bootstrap: np.ndarray,
      factor_cov: pd.DataFrame,
      beta_estimation_var: pd.DataFrame,
      min_assets: int = 2,
      scenario_idx=None,
    ) -> np.ndarray:
        N, K = unit.shape
        if not 1 <= min_assets <= N:
            raise ValueError("min_assets must be in [1,N]")

        if not beta_adjusted.index.equals(unit.index) or not beta_adjusted.columns.equals(unit.columns):
            raise ValueError("beta_adjusted and unit must have identical (Asset,Factor) axes")

        if coefs.shape[1:] != (N, K):
            raise ValueError("coefs must be (B,N,K) with unit asset/factor ordering")

        if quality.shape != (N,) or not np.isfinite(quality).all() or np.any(quality < 0):
            raise ValueError("quality must be nonnegative finite (N,)")

        if not isinstance(factor_cov, pd.DataFrame):
            raise ValueError("factor_cov must be a DataFrame indexed by Factor")

        sigma = factor_cov.loc[unit.columns, unit.columns].to_numpy(dtype=float)

        delta = np.asarray(factor_scenarios, dtype=float)
        a = np.asarray(scenario_scale, dtype=float)

        if delta.ndim != 2 or delta.shape[1] != K or a.shape != (len(delta),):
            raise ValueError("factor_scenarios (S,K) and scenario_scale (S,) required")

        if not np.isfinite(delta).all() or not np.isfinite(a).all() or np.any(a <= 0):
            raise ValueError("Scenario factor shocks must be finite; scales must be >0")

        S_all = len(delta)
        if S_all == 0:
            raise ValueError("At least one scenario is required")

        if scenario_idx is not None:
            selected = np.asarray(scenario_idx)
            if selected.dtype == bool:
                if selected.shape != (S_all,):
                    raise ValueError("Boolean scenario_idx must be (S,)")
            else:
                selected = np.atleast_1d(selected).astype(int)
                if selected.ndim != 1 or np.any((selected < 0) | (selected >= S_all)):
                    raise ValueError("scenario_idx contains invalid positions")
            delta, a = delta[selected], a[selected]

        if len(delta) == 0:
            raise ValueError("At least one selected scenario is required")

        S = len(delta)

        p_s = np.full(S, 1.0 / S)

        beta_array = beta_adjusted.to_numpy(dtype=float)
        r_hat = beta_array @ delta.T

        baseline = self._validate_selection_result(
            result=self.optim_select_assets(
                unit=unit, quality=quality, r_hat=r_hat, n_select=min_assets
            ),
            N=N, K=K, S=S, m=min_assets,
        )

        opt_selection = baseline[0].copy()
        prev = baseline

        for m in range(min_assets + 1, N + 1):
          current = self._validate_selection_result(
              result=self.optim_select_assets(
                  unit=unit, quality=quality, r_hat=r_hat, n_select=m
              ),
              N=N, K=K, S=S, m=m,
          )

          excess, redundancy, marginal_gain = self._get_excess_coverage(
              unit=unit,
              betas=beta_array,
              factor_cov=sigma,
              quality=quality,
              factor_scenarios=delta,
              scenario_scale=a,
              p_s=p_s,
              prev_result=prev,
              current_result=current,
          )

          uncertainty = self._get_uncertainty(
              coefs=coefs,
              factors_boot=factor_bootstrap,
              unit=unit,
              quality=quality,
              m=m,
              factor_scenarios=delta,
              scenario_scale=a,
              p_s=p_s,
              beta_estimation_var=beta_estimation_var,
          )

          if self.fp.debug:
            print(
              f"m={m}: marginal_gain={marginal_gain:.6g}, "
              f"redundancy={redundancy:.6g}, excess={excess:.6g}, "
              f"bootstrap_sd={uncertainty:.6g}"
            )

          if excess <= uncertainty:
              break

          opt_selection = current[0].copy()
          prev = current

        return opt_selection

    def select_assets(
      self,
      returns: pd.DataFrame,
      factors: pd.DataFrame,
      quality: pd.Series | np.ndarray,
      factor_scenarios: pd.DataFrame | np.ndarray,
      scenario_scale: np.ndarray,
      min_assets: int = 2,
      scenario_idx=None,
    ) -> np.ndarray:
      (
          beta_adjusted, beta_change, summary, coefs,
          factor_bootstrap, aligned_factor_window, est_var,
      ) = self._get_factor_exposure_features(returns=returns, factors=factors)

      assets, fnames = beta_adjusted.index, beta_adjusted.columns

      if isinstance(quality, pd.Series):
          quality_array = quality.reindex(assets).to_numpy(dtype=float)

      else:
          quality_array = np.asarray(quality, dtype=float)

      if quality_array.shape != (len(assets),) or not np.isfinite(quality_array).all():
          raise ValueError("quality must align with surviving asset columns")

      if isinstance(factor_scenarios, pd.DataFrame):
          missing = fnames.difference(factor_scenarios.columns)
          if len(missing):
              raise ValueError(f"Missing scenario factors: {missing.tolist()}")
          factor_scenarios = factor_scenarios.loc[:, fnames].to_numpy(dtype=float)

      _, unit, _ = self._risk_exposure_similarity(
          betas=beta_adjusted, factors=aligned_factor_window
      )
      factor_cov = aligned_factor_window.cov()

      return self._select_opt_n_assets(
          beta_adjusted=beta_adjusted,
          unit=unit,
          quality=quality_array,
          coefs=coefs,
          factor_scenarios=factor_scenarios,
          scenario_scale=scenario_scale,
          factor_bootstrap=factor_bootstrap,
          factor_cov=factor_cov,
          beta_estimation_var=est_var,
          min_assets=min_assets,
          scenario_idx=scenario_idx,
      )

    def run_factor_filter(
      self,
      returns: pd.DataFrame,
      factors: pd.DataFrame,
      quality: pd.Series | np.ndarray,
      factor_scenarios: pd.DataFrame | np.ndarray,
      scenario_scale: np.ndarray,
      min_assets: int = 2,
      scenario_idx=None,
    ) -> pd.Series:
      if not isinstance(quality, pd.Series):
        quality = pd.Series(np.asarray(quality, dtype=float), index=returns.columns)

      screened = self._time_horizon_filter(returns=returns)

      selection = self.select_assets(
          returns=screened,
          factors=factors,
          quality=quality,
          factor_scenarios=factor_scenarios,
          scenario_scale=scenario_scale,
          min_assets=min_assets,
          scenario_idx=scenario_idx,
      )

      return pd.Series(selection, index=screened.columns, name="Weight")